In [ ]:
import numpy as np
import pandas as pd

In [ ]:

np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

시나리오 1 — 검증 → 병합
(1) 검증: 룩업 표의 키가 유일한가, 주문의 키가 모두 매칭되는가


In [ ]:

print("[병합 전 검증]")
print("  customers 키 중복:", cust["customer_id"].duplicated().sum(), "건")
print("  products  키 중복:", prod["product_id"].duplicated().sum(), "건")
print("  매칭 안 되는 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("  매칭 안 되는 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")

df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
display(df.head(3))

In [ ]:
# 시나리오 2 — 월별 KPI
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_kpi = (
    df.groupby("month")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
        구매고객수=("customer_id", "nunique"),   # nunique: 고유 고객 수
    )
    .round(0)
)
# 전월 대비 매출 증감률(%)도 추가 — 경영 보고서의 단골 지표
monthly_kpi["매출증감률(%)"] = (monthly_kpi["총매출"].pct_change() * 100).round(1)

print("[월별 KPI 요약표]")
display(monthly_kpi)

In [ ]:
# 시나리오 3-1 — 월 × 카테고리 매출 교차표 (pivot_table)
month_category = pd.pivot_table(
    df, index="month", columns="category", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
).round(0)
print("[월 × 카테고리 매출 교차표]")
display(month_category)

# 시나리오 3-2 — 각 주문이 '그 달 전체 매출'에서 차지하는 비중 (transform)
df["month_total"] = df.groupby("month")["amount"].transform("sum")
df["share_in_month(%)"] = (df["amount"] / df["month_total"] * 100).round(3)
print("\n[주문별 월 매출 점유율 — 앞 5행]")
display(df[["order_id", "month", "amount", "share_in_month(%)"]].head())

In [ ]:
# 핵심 숫자 자동 추출 (보고서 문장에 채워 넣기 위함)
best_month = monthly_kpi["총매출"].idxmax()
best_month_sales = monthly_kpi["총매출"].max()
top_category = df.groupby("category")["amount"].sum().idxmax()
top_region = df.groupby("region")["amount"].sum().idxmax()
overall_aov = df["amount"].mean()

print("자동 추출된 핵심 숫자")
print(f"  최대 매출 월   : {best_month} ({best_month_sales:,.0f}원)")
print(f"  최대 매출 카테고리: {top_category}")
print(f"  최대 매출 지역 : {top_region}")
print(f"  전체 객단가    : {overall_aov:,.0f}원")